# DORA Metrics for Failure Only

## Env

In [1]:
print("Kernel is working.")

Kernel is working.


In [2]:
from dash import dcc
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

## Data Ingestion

**Output:** pandas dataframe df_fail

In [12]:
# Useful reference: <https://pandas.pydata.org/docs/getting_started/intro_tutorials/09_timeseries.html>
raw_file_path = '/home/lnx_workspaces/bpp_projects/bpp_module2_project/doraview/data/json/change_fail.json'

# Try reading with explicit encoding and error handling
try:
    df_fail = pd.read_json(raw_file_path, encoding='utf-8', convert_dates=["detected_at"]).sort_values(by=["detected_at"])
    print("\nDataframe info:")
    print(df_fail.info())
    print("\nFirst 5 rows:")
    print(df_fail.head())
except Exception as e:
    print(f"Error: {str(e)}")


Dataframe info:
<class 'pandas.core.frame.DataFrame'>
Index: 7 entries, 4 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   failure_id      7 non-null      object             
 1   application_id  7 non-null      object             
 2   deployment_id   7 non-null      object             
 3   failed          7 non-null      bool               
 4   rollback        7 non-null      bool               
 5   failure_reason  7 non-null      object             
 6   detected_at     7 non-null      datetime64[ns, UTC]
dtypes: bool(2), datetime64[ns, UTC](1), object(4)
memory usage: 350.0+ bytes
None

First 5 rows:
      failure_id application_id deployment_id  failed  rollback  \
4  fail_331a648f         app003   df_d6b6cdf5    True      True   
0  fail_3f89e627         app002   df_a29aae73    True     False   
6  fail_839d682c         app003   df_f07e8344    True     False   
3  fail_d

## Data Manipulation

**Output:** Updates to pandas dataframe *df_fail* for use by figures.
- New aggregated month datetime column.
- New Moving Average columns for plotting (SMA, EMA, CMA)

In [13]:
df_fail["month"] = df_fail["detected_at"].dt.month

df_fail.head(100)

,failure_id,application_id,deployment_id,failed,rollback,failure_reason,detected_at,month
4,fail_331a648f,app003,df_d6b6cdf5,True,True,config error,2025-07-16 01:40:00+00:00,7
0,fail_3f89e627,app002,df_a29aae73,True,False,dependency issue,2025-07-19 08:43:00+00:00,7
6,fail_839d682c,app003,df_f07e8344,True,False,runtime exception,2025-08-03 21:25:00+00:00,8
3,fail_d229b031,app003,df_d7800254,True,False,runtime exception,2025-08-06 14:01:00+00:00,8
1,fail_2d83092d,app002,df_883f7d05,True,False,runtime exception,2025-08-16 16:08:00+00:00,8
2,fail_d3275dfc,app002,df_df253d66,True,False,runtime exception,2025-09-11 08:26:00+00:00,9
5,fail_d7b798ea,app003,df_a72a3a42,True,False,runtime exception,2025-09-27 01:22:00+00:00,9


## Figures

### ChatGPT - Change Failure Rate (CFR)

**Best Graph Type:** Stacked Bar Chart (success vs. failed deployments)

**Why:**

*CFR is a percentage:*

- failed deployments ÷ total deployments

**Stacked bars clearly show:**

- Volume of deployments
- Proportion that failed vs. succeeded
- How stability changes over time
- Visual impact of process or team improvements

**Optional secondary views:**

- Donut/Pie Chart for a single period (e.g., this month)
- Line Chart for the CFR % trend over time

For dashboards, a percentage KPI tile + stacked bars is very common.

### Multi-Group Display

In [18]:
df_fail_grouped = df_fail.groupby([
	"application_id",
	"month",
	])["month"].count().reset_index(name='count')

df_fail.head(100)

,failure_id,application_id,deployment_id,failed,rollback,failure_reason,detected_at,month
4,fail_331a648f,app003,df_d6b6cdf5,True,True,config error,2025-07-16 01:40:00+00:00,7
0,fail_3f89e627,app002,df_a29aae73,True,False,dependency issue,2025-07-19 08:43:00+00:00,7
6,fail_839d682c,app003,df_f07e8344,True,False,runtime exception,2025-08-03 21:25:00+00:00,8
3,fail_d229b031,app003,df_d7800254,True,False,runtime exception,2025-08-06 14:01:00+00:00,8
1,fail_2d83092d,app002,df_883f7d05,True,False,runtime exception,2025-08-16 16:08:00+00:00,8
2,fail_d3275dfc,app002,df_df253d66,True,False,runtime exception,2025-09-11 08:26:00+00:00,9
5,fail_d7b798ea,app003,df_a72a3a42,True,False,runtime exception,2025-09-27 01:22:00+00:00,9


In [ ]:
# display dataframe as figure
fig_month_multi_bar = px.bar(
	data_frame=df_fail_grouped,
	title="Total Monthly Deployments",	# Label for the figure.
	x="month",							# Column for use on x-axis
	y="count",							# Column for use on y-axis
	color="failed"						# Column for use as colour differentiation
	barmode="group",					# Group bars together
	subtitle="All Applications"
)

fig_month_multi_bar.update_yaxes(
	title_text="Failure Count"		# Update title for y-axis
)

fig_month_multi_bar.update_xaxes(
	title_text="Deployment Month",		# Update title for x-axis
	tickvals=list(range(1,13)),			# Specify values of x-axis
	ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']	# Allocate text to tick values.
)

fig_month_multi_bar.show()

ValueError: Value of 'color' is not the name of a column in 'data_frame'. Expected one of ['application_id', 'month', 'count'] but received: failed